# Reconstructing raw MRI data

Relevant tutorials / exercises for MRI raw data recon:
https://github.com/SyneRBI/SIRF-Exercises/tree/master/notebooks/MR

## Dataset

Example data set contains axial proton density-weighted knee images with fat saturation.

Data import / prep

- Prior to this Christoph converted raw data from MRI scanner (.dat format, vendor-specific) to ISMRMD .h5 format using ISMRMD method.
- Raw MRI data is loaded into a CIL/SIRF AcquisitionData container
- Raw data is pre-processed through gadgetron gadgets before use. E.g. Removing oversampling along the readout direction, ensuring noise between receiver coils is not correlated, and padding readouts to compensate for partial echo acquisitions. (See [SIRF Exercises](https://github.com/SyneRBI/SIRF-Exercises/blob/master/notebooks/MR/a_fully_sampled.ipynb) for further info)

In [ ]:
# Imports etc

'''
GRAPPA reconstruction with an iterative algorithm from CIL: illustrates
the use of AcquisitionModel in CIL optimisation 

Usage:
  grappa_and_cil.py [--help | options]

Options:
  -f <file>, --file=<file>    raw data file
                              [default: simulated_MR_2D_cartesian_Grappa2.h5]
  -p <path>, --path=<path>    path to data files, defaults to data/examples/MR
                              subfolder of SIRF root folder
'''

## CCP PETMR Synergistic Image Reconstruction Framework (SIRF)
## Copyright 2015 - 2019 Rutherford Appleton Laboratory STFC.
## Copyright 2015 - 2019 University College London.
##
## This is software developed for the Collaborative Computational
## Project in Positron Emission Tomography and Magnetic Resonance imaging
## (http://www.ccppetmr.ac.uk/).
##
## Licensed under the Apache License, Version 2.0 (the "License");
##   you may not use this file except in compliance with the License.
##   You may obtain a copy of the License at
##       http://www.apache.org/licenses/LICENSE-2.0
##   Unless required by applicable law or agreed to in writing, software
##   distributed under the License is distributed on an "AS IS" BASIS,
##   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
##   See the License for the specific language governing permissions and
##   limitations under the License.
    

import sirf
from sirf.Utilities import existing_filepath
from sirf.Utilities import error
from sirf.Utilities import show_3D_array
from sirf.Gadgetron import examples_data_path
from sirf.Gadgetron import AcquisitionData, ImageData
from sirf.Gadgetron import AcquisitionModel
from sirf.Gadgetron import AcquisitionDataProcessor
from sirf.Gadgetron import CartesianGRAPPAReconstructor, FullySampledReconstructor
from sirf.Gadgetron import CoilSensitivityData
from sirf.Gadgetron import preprocess_acquisition_data

# Import algorithms, operators and functions from CIL optimisation module

from cil.plugins.ccpi_regularisation.functions import FGP_TV # TGV, LLT_ROF, Diff4th
from cil.framework import DataContainer as cilDataContainer
from cil.optimisation.algorithms import GD, FISTA, PDHG, CGLS
from cil.optimisation.operators import BlockOperator, GradientOperator,\
                                       GradientOperator, LinearOperator, WaveletOperator
from cil.optimisation.functions import IndicatorBox, MixedL21Norm, L2NormSquared, \
                                       BlockFunction, L1Norm, LeastSquares, \
                                       OperatorCompositionFunction, TotalVariation, Function, L1Sparsity, FunctionOfAbs, ZeroFunction
from cil.utilities.display import show2D
from cil.utilities.jupyter import islicer
from cil.processors import Slicer

# External imports
import numpy as np
import time
import math
import matplotlib.pyplot as plt
import os
import gc
import torch
import deepinv

In [ ]:
# Getting data

# process command-line options

fname_ai = '1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5'    # Under-sampled (to be constructed using DL methods)
fname_full = '1meas_MID00614_FID129152_CONVENTIONAL_RECON_SEQD_GF2_AX_RL_mod.h5'  # Fully sampled
data_path = '/home/jovyan/work/data/h5'

input_file = os.path.join(data_path, fname_full)
print (input_file)

# acquisition data will be read from an HDF file input_data

acq_data_full = AcquisitionData(input_file)

# Pre-processing and reconstruction

acq_data = preprocess_acquisition_data(acq_data_full)

## Fully sampled data

In MR the dimensions of the image data are altered by the reconstruction process. This means we need to carry out an example reconstruction to get all information about the image.

In [ ]:
# Reconstruct fully sampled k-space data

recon = FullySampledReconstructor()
recon.set_input(acq_data)
recon.process()


full_recon = np.abs(recon.get_output().asarray(), dtype=np.float32)
print(type(full_recon), full_recon.dtype)

# Rotate image to same orientation as it appears on hospital PACS
full_recon_rotated = np.rot90(full_recon, axes=(2, 1))

show2D(full_recon_rotated)

## Under-sampled data
(requires de-noising / AI reconstruction)

- Raw MRI data is loaded into a CIL AcquisitionData container.
- Coil sensitivity maps are calculated. Map tells us pixel by pixel how much that coil contributes to the signal at that location.

In [92]:
## Input file contains downsampled raw data in h5 format (k-space data + sensitivity maps + metadata +..??)

input_file = os.path.join(data_path, fname_ai)

# DataContainer for holding raw data

acq_data_ai = AcquisitionData(input_file)   

# Prep data for reconstruction

acq_data_ai = preprocess_acquisition_data(acq_data_ai)

#Calculate coil sensitivity maps from raw k-space data using ESPIRiT (uses central, fully sampled region of k-space)

csm = CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data_ai)


reading from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5 using ignore mask 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 0000 

Started reading acquisitions from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5
0%..10%..20%..30%..40%..50%..60%..70%..80%..90%..100%..
Finished reading acquisitions from /home/jovyan/work/data/h5/1meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL_mod.h5
ignoring acquisition 0
Message received with ID: 5
Input stream has terminated


The acquisition model in MRI is:
```
d_c = F · S_c · x
```
x is the image you want to reconstruct
S_c is the sensitivity map for coil c (a pointwise multiplication)
F is the Fourier transform (or NUFFT for non-Cartesian sampling)
d_c is the measured k-space data for coil c

In [86]:
# Acquisition model maps k-space to image space

E = AcquisitionModel(acqs=acq_data_ai, imgs=csm)   # Why?
E.set_coil_sensitivity_maps(csm)
x_inverse = E.inverse(acq_data_ai)

## 1. FISTA, no regulariser

In [87]:
# Reconstruction

# Define forward model E and loss function f for iterative reconstruction

E = AcquisitionModel(acqs=acq_data_ai, imgs=x_inverse)
E.set_coil_sensitivity_maps(csm)

# Use the result of the inverse as our starting point

x_init = x_inverse.clone()

# Data fidelity term f (objective/loss function) is least squares between Ex and y

f = LeastSquares(E, acq_data_ai, c=1)

# No regularisation used

G = ZeroFunction()

# Set up FISTA

fista = FISTA(initial=x_init.fill(0.0), f=f, g=G)
fista.update_objective_interval = 5

# Run FISTA for least squares

fista.run(10)

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# Display reconstructed images

# Magnitude image

ai_recon = np.abs(fista.solution.asarray(), dtype=np.float32)

# Normalise pixel vals (?Adjusts window width?)

def normalise(data):
    return (data - data.min())/(data.max()-data.min())

fn_recon = normalise(full_recon)
ain_recon = normalise(ai_recon)

# Rotate to same orientation as images appear on hospital PACS

fn_recon_rotated = np.rot90(fn_recon, axes=(2, 1))
ain_recon_rotated = np.rot90(ain_recon, axes=(2, 1))
show2D([fn_recon_rotated, ain_recon_rotated], slice_list=(0, 6), fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 95))) #(?Also adjusts window width?)

## 2. FISTA with FGP TV regulariser

- Regulariser is Fast Gradient Projection-based Total Variation
- Total cost = Data fidelity term + (α × Total Variation term)

In [ ]:
# Make regularisation function using CCPi-Regularisation toolkit.
# Alpha describes regularisation strength (how strongly TV enforced)

G = FGP_TV(alpha = 0.3)

algo_tv = FISTA(initial=x_init.fill(0.0), f=f, g=G, update_objective_interval=1)

print("algo configured")

algo_tv.run(70)

In [ ]:
# Display reconstructed images

tv_recon = np.abs(algo_tv.solution.asarray(), dtype=np.float32)

tv_recon = normalise(tv_recon)

# Rotate images to same orientation as on hosp PACS

tv_recon_rotated = np.rot90(tv_recon, axes=(2, 1))

show2D([el[:,250:400, 180:320] for el in [fn_recon_rotated, tv_recon_rotated]],
       title=['Fully sampled', 'LS+TV'],
       slice_list=(0, 7), 
       fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 99)),
      num_cols=2)

show2D([el[:,70:430, 70:430] for el in [fn_recon_rotated, tv_recon_rotated]],
       title=['Fully sampled', 'LS+TV'],
       slice_list=(0, 7), 
       fix_range=(np.percentile(fn_recon, 5), np.percentile(fn_recon, 99)),
      num_cols=2)

## 3. FISTA with ?wavelet regulariser?

In [64]:
# Edo wavelets

w = WaveletOperator(x_init, wname="bior4.4", level=1)
R_W = FunctionOfAbs(
    L1Sparsity(w)
)

/opt/conda/lib/python3.12/site-packages/cil/optimisation/functions/L1Sparsity.py:54: UserWarning: Invalid operator: `<cil.optimisation.operators.WaveletOperator.WaveletOperator object at 0x7f4fc038a3c0>`. L1Sparsity is properly defined only for orthogonal operators!
  warnings.warn(


In [68]:
# Edo wavelets

x_init.geometry = x_init
algo_w = FISTA(x_init, f=f, g=R_W)
algo_w.run(10)


AttributeError: 'ImageData' object has no attribute 'geometry'

In [ ]:
np.save??

In [22]:
# External imports
import numpy as np
import time
import math
import matplotlib.pyplot as plt
import os
import gc
import torch

## 4. FISTA with plug and play de-noiser

In [88]:
# Imports
import deepinv as dinv
from cil.optimisation.functions import Function
from denoiser_proximal import DenoiserProximal

In [73]:
# Set the device to be used by torch

device = dinv.utils.get_device()

# Choose trained PyTorch de-noiser

denoiser = deinv.models.DnCNN(in_channels=1, out_channels=1, pretrained='download', device=device)

model = dinv.optim.DPIR(sigma=0.1, denoiser=denoiser, device=device)

x_hat = model(y, physics)

#  Neural networks should usually be set to evaluation mode before use

trained_DnCNN.eval()


Downloading: "https://huggingface.co/deepinv/dncnn/resolve/main/dncnn_sigma2_lipschitz_gray.pth?download=true" to /home/jovyan/.cache/torch/hub/checkpoints/dncnn_sigma2_lipschitz_gray.pth







  0% 0.00/2.55M [00:00<?, ?B/s]




100% 2.55M/2.55M [00:00<00:00, 19.8MB/s]


DnCNN(
  (in_conv): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv_list): ModuleList(
    (0-17): 18 x Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (out_conv): Conv2d(64, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (nl_list): ModuleList(
    (0-18): 19 x ReLU()
  )
)

In [91]:
# FISTA solution with TV regulariser (type ImageData)

ai_recon = algo_tv.solution

In [ ]:

# Set de-noiser as regulariser

regulariser = alpha*DenoiserProximal(denoiser=trained_DRUNet, device='cpu')

f = LeastSquares(E, acq_data_ai, c=1)

# Set starting point (is same as starting point used above)



# Set up FISTA with dncnn denoiser

algo_fista_dncnn = FISTA(f=f, 
                  g=regulariser, 
                  initial=x0,
                  update_objective_interval = 5)

print("algo configured")
